# Next_Word_Predictor

# Dependency Install

In [54]:
!pip install python-docx

# Import Libraries

In [55]:
import pandas as pd
import numpy as np

In [56]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

In [57]:
from docx import Document

# Load Data

In [58]:
doc = Document('Plantive-QnA.docx')
doc

In [59]:
document = []
for para in doc.paragraphs:
    if para.text != "":
        document.append(para.text)
document

['Gardener App – 25 Detailed Questions & Answers',
 '1. What is the main purpose of this gardening application?',
 'The purpose of this application is to help users easily purchase plants and gardening-related products/services while also monitoring the health of their plants in real-time. It acts as a complete digital gardening assistant, guiding users from plant selection to plant care.',
 '2. Who is the target audience for this application?',
 'The app is designed for home gardeners, rooftop gardeners, plant hobbyists, beginners, and anyone who wants to grow and maintain plants with smart monitoring and expert guidance.',
 '3. What types of items can users purchase from the marketplace?',
 'Users can purchase various items such as live plants, pots, fertilizers, seeds, soil, gardening tools, accessories, and any plant-care-related equipment.',
 '4. What kinds of service packages are offered in the app?',
 'The app includes admin-created service packages like plant combo deals, rooft

In [60]:
document[0]

'Gardener App – 25 Detailed Questions & Answers'

# Tokenize Data

In deep learning, particularly in Natural Language Processing (NLP), a tokenizer is a crucial component that breaks down raw text into smaller units called tokens. These tokens, which can be words, subwords, or characters, are then converted into numerical IDs that machine learning models can process.

In [61]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(document)

len(tokenizer.word_index)

395

# Make Label

In [62]:
# Read Line by Line
# Convert that Line to Numaric
# For First 2 word, label will be 3 and first 3, label will be 4th

input_sequences = []
for line in document:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

In [63]:
document[2]

'The purpose of this application is to help users easily purchase plants and gardening-related products/services while also monitoring the health of their plants in real-time. It acts as a complete digital gardening assistant, guiding users from plant selection to plant care.'

In [64]:
input_sequences[100]

[167, 14, 168, 22, 92, 8, 5, 52, 42, 1]

In [65]:
len(input_sequences), len(document)

(944, 53)

# Add Padding

In [66]:
max_len = max([len(x) for x in input_sequences])
max_len

44

In [67]:
padded_input = pad_sequences(sequences=input_sequences, maxlen=max_len, padding='pre')
padded_input

array([[  0,   0,   0, ...,   0, 146,   4],
       [  0,   0,   0, ..., 146,   4, 147],
       [  0,   0,   0, ...,   4, 147,  81],
       ...,
       [  0,   0,   0, ...,  28, 391, 392],
       [  0,   0,   0, ...,   0, 393, 394],
       [  0,   0,   0, ..., 393, 394, 395]], dtype=int32)

# X, y Separate

In [68]:
X = padded_input[:, :-1]
y = padded_input[:, -1]

In [69]:
X.shape, y.shape

((944, 43), (944,))

# Change `Y` to categorical

In [70]:
y = to_categorical(y, num_classes=len(tokenizer.word_index)+1)
y.shape

(944, 396)

# LSTM Model

In [71]:
model = Sequential()
model.add(Input(shape=(max_len-1,)))
model.add(Embedding(input_dim=len(tokenizer.word_index)+1, output_dim=100))

model.add(LSTM(units=150, return_sequences=True))
model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(LSTM(units=100))
model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(Dense(units=100, activation='relu'))
model.add(Dense(units=len(tokenizer.word_index)+1, activation='softmax'))

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 43, 100)        │        39,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 43, 150)        │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 43, 150)        │           600 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 43, 150)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 396)            │        39,996 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 341,696 (1.30 MB)

 Trainable params: 341,196 (1.30 MB)

 Non-trainable params: 500 (1.95 KB)

## Model Training

In [72]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [73]:
history = model.fit(
    x=X,
    y=y,
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 9s 140ms/step - accuracy: 0.0210 - loss: 5.9867 - val_accuracy: 0.0370 - val_loss: 5.9402
Epoch 2/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 4s 87ms/step - accuracy: 0.0504 - loss: 5.3555 - val_accuracy: 0.0529 - val_loss: 5.9048
Epoch 3/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 86ms/step - accuracy: 0.0842 - loss: 4.9162 - val_accuracy: 0.0529 - val_loss: 5.8935
Epoch 4/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 105ms/step - accuracy: 0.0720 - loss: 4.6208 - val_accuracy: 0.0529 - val_loss: 5.8774
Epoch 5/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 5s 82ms/step - accuracy: 0.1101 - loss: 4.3078 - val_accuracy: 0.0370 - val_loss: 5.8537
Epoch 6/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 85ms/step - accuracy: 0.1169 - loss: 4.0231 - val_accuracy: 0.0265 - val_loss: 5.8428
Epoch 7/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 89ms/step - accuracy: 0.1440 - loss: 3.7091 - val_accuracy: 0.0317 - val_loss: 5.8342
Epoch 8/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 106ms/step - accuracy: 0.2027 - loss: 3.4424 - val_accuracy:

# Predict

In [86]:
# Yes. Admins can create new plant profiles, update care instructions, add new service packages, upload product details, and modify existing data in real-time.

text = " Yes. Admins can"

for i in range(7):
    # Tokenize
    token_list = tokenizer.texts_to_sequences([text])[0]

    # Padding
    token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')

    # Predict
    predicted = np.argmax(model.predict(token_list), axis=-1)


    output_word = ""
    for word, index in tokenizer.word_index.items():
        if index == predicted:
            output_word = word
            print(text)
            break
    text += " " + output_word

print(text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
 Yes. Admins can
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
 Yes. Admins can create
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
 Yes. Admins can create new
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
 Yes. Admins can create new plant
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
 Yes. Admins can create new plant profiles
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
 Yes. Admins can create new plant profiles update
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
 Yes. Admins can create new plant profiles update care
 Yes. Admins can create new plant profiles update care instructions
